# How does a Bike-share navigate speedy success?

## Overview

This notebook demonstrates end-to-end data analysis workflow focusing on:
- Data preparation and quality assessment
- Exploratory analysis and pattern discovery
- Visualization of key insights
- Actionable recommendations

**Key question:** What distinguishes casual riders from annual members, and how can we convert more casual riders into memberships?

**Data source:** [Public Divvy trip data](https://divvy-tripdata.s3.amazonaws.com/index.html)

**Privacy and reproducibility:**
- This is an aggregated analysis that uses only trip-level metadata with no personal/individual identifiers.
- Raw data files are expected under `data/raw/` (this directory is git-ignored due to file size). Download the datasets from the source URL above and store each file using the following structure: `data/raw/{YYYYMM}-divvy-tripdata/{YYYYMM}-divvy-tripdata.csv`, where `YYYYMM` represents the year and month (e.g., `202601`).
- All transformations are documented in this notebook.
- Processed data outputs are saved to `data/processed/` for downstream use.

In [1]:
"""Demonstrates a complete data analysis pipeline."""

import pandas as pd
from pathlib import Path

## Load data

Load up to the most recent 12 months of trip data, stored as individual CSV files in `data/raw/`.

In [2]:
# Get list of raw data files sorted by date (up to 12 most recent months)
base = Path('../data/raw')
files = sorted(base.glob("*-divvy-tripdata/*.csv"))[-12:]

# Load files as a list of dataframes
dfs = [pd.read_csv(file) for file in files]

# Extract months from file names
months_pretty = [
    pd.to_datetime(f.parent.name[:6], format="%Y%m").strftime("%B %Y")
    for f in files
]

print(f"Loaded {len(dfs)} month(s) data: {', '.join(months_pretty)}.")

Loaded 12 month(s) data: April 2025, May 2025, June 2025, July 2025, August 2025, September 2025, October 2025, November 2025, December 2025, January 2026, February 2026, March 2026.


## Standardize schema

Standardize column names, column types, `member_casual` labels, and columns across files.

In [3]:
# Fix most common inconsistencies (if any)
updated_dfs = []
for df in dfs:
    # Standardize column names
    df = df.rename(columns={
        'trip_id': 'ride_id', 'usertype': 'member_casual',
        'start_time': 'started_at', 'end_time': 'ended_at',
        'from_station_id': 'start_station_id', 'to_station_id': 'end_station_id',
        'from_station_name': 'start_station_name', 'to_station_name': 'end_station_name'
    })

    # Standardize column dtypes
    df['ride_id'] = df['ride_id'].astype(str)
    df['started_at'] = pd.to_datetime(df['started_at'], utc=True)
    df['ended_at'] = pd.to_datetime(df['ended_at'], utc=True)

    # Standardize `member_casual` labels
    df['member_casual'] = df['member_casual'].replace({
        'Subscriber': 'member', 'Customer': 'casual'
    })
    updated_dfs.append(df)
dfs = updated_dfs

# Standardize columns across files
required_columns = {
    'ride_id', 'member_casual', 'started_at', 'ended_at',
    'start_station_id', 'end_station_id',
    'start_station_name', 'end_station_name'
}
for i, df in enumerate(dfs):
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"File {files[i]} missing columns: {missing_columns}")

print("Standardization of schema completed across data sources.")

Standardization of schema completed across data sources.


## Combine datasets

Bind required rows from multiple dataframes into single dataframe and remove exact duplicates.

In [4]:
# Display floats with commas and up to two decimals (no scientific notation)
pd.options.display.float_format = "{:,.2f}".format

all_trips = pd.concat(
    [df[list(required_columns)] for df in dfs],
    ignore_index=True
)
count_before_dedupe = len(all_trips)
all_trips = all_trips.drop_duplicates()
print(f"Deduplicated {count_before_dedupe - len(all_trips)} rows.")
print(f"Total trips: {len(all_trips):,}.")

Deduplicated 0 rows.
Total trips: 5,620,544.


## Display summary statistics

Display summary statistics like row count, columns, data types, and missing values.

In [5]:
print("=== Dataset Overview ===")
print(f"Shape: {all_trips.shape[0]:,} rows x {all_trips.shape[1]:,} columns.")

print("\n=== Data Types ===")
print(all_trips.dtypes)

print("\n=== Missing Values ===")
missing = all_trips.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].map(lambda x: f"{x:,}"))
else:
    print("No missing values detected.")

=== Dataset Overview ===
Shape: 5,620,544 rows x 8 columns.

=== Data Types ===
end_station_id                        str
member_casual                         str
start_station_name                    str
start_station_id                      str
started_at            datetime64[us, UTC]
end_station_name                      str
ride_id                               str
ended_at              datetime64[us, UTC]
dtype: object

=== Missing Values ===
end_station_id        1,259,214
start_station_name    1,194,952
start_station_id      1,194,952
end_station_name      1,259,214
dtype: str


## Address missing values

Remove rows with missing values in required columns like `ride_id`, `end_station_id`, and `start_station_id` (data quality).

In [6]:
# Drop rows with missing values
before = len(all_trips)
all_trips = all_trips.dropna(subset=['ride_id', 'end_station_id', 'start_station_id'])
print(f"{before - len(all_trips):,} trips removed with missing values.")

1,881,299 trips removed with missing values.


## Derive new features

- Compute ride duration in seconds.
- Compute time features like month and day.

In [7]:
# Compute ride duration
all_trips['ride_duration'] = (all_trips['ended_at'] - all_trips['started_at']).dt.total_seconds()

# Derive time features
all_trips['started_month'] = all_trips['started_at'].dt.month_name().str[:3]
all_trips['started_day'] = all_trips['started_at'].dt.day_name().str[:3]

# Convert time features to ordered categorical for consistent ordering
for col, col_order in {
    'started_month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    'started_day': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
}.items():
    pd.Categorical(all_trips[col], categories=col_order, ordered=True)

print("Feature creation completed.")

print("\n=== Basic Statistics ===")
print(all_trips.describe(include='all'))

Feature creation completed.

=== Basic Statistics ===
       end_station_id member_casual                 start_station_name  \
count         3739245       3739245                            3739245   
unique           3246             2                               1856   
top          CHI01747        member  DuSable Lake Shore Dr & Monroe St   
freq            52806       2407707                              36325   
mean              NaN           NaN                                NaN   
min               NaN           NaN                                NaN   
25%               NaN           NaN                                NaN   
50%               NaN           NaN                                NaN   
75%               NaN           NaN                                NaN   
max               NaN           NaN                                NaN   
std               NaN           NaN                                NaN   

       start_station_id                        started_at

## Remove irrelevant trips

- Trips with negative or zero ride durations.
- Filter out extreme outliers in ride durations for EDA visuals and summary stats.
- Filter out trips out of expected date bounds

In [8]:
# Drop rows with negative or zero ride durations
before = len(all_trips)
all_trips = all_trips[all_trips['ride_duration'] > 0].copy()
print(f"{before - len(all_trips):,} trips removed with negative or zero ride duration.")

# Filter out top 1% durations to limit extreme outliers
before = len(all_trips)
duration_threshold = all_trips['ride_duration'].quantile(0.99)
all_trips = all_trips[
  all_trips['ride_duration'] < duration_threshold
].copy()
print(f"{before - len(all_trips):,} trips removed with top 1% ride durations.")

# Drop rows out of expected date bounds from loaded months
before = len(all_trips)
min_date = pd.to_datetime(files[0].parent.name[:6], format="%Y%m", utc=True)
max_date = (
    pd.to_datetime(files[-1].parent.name[:6], format="%Y%m", utc=True)
    + pd.offsets.MonthEnd()
    + pd.Timedelta(days=1)
)
all_trips = all_trips[(all_trips['started_at'] >= min_date) &
                      (all_trips['started_at'] < max_date)]
print(f"{before - len(all_trips):,} trips removed as started out of expected date bounds.")

print(f"Total trips after cleaning: {len(all_trips):,}.")

19 trips removed with negative or zero ride duration.
37,393 trips removed with top 1% ride durations.
11 trips removed as started out of expected date bounds.
Total trips after cleaning: 3,701,822.


## Statistical summary post-cleaning

Review row counts, columns, data types, and missing values after data cleaning and standardization.

In [9]:
print("=== Dataset Overview ===")
print(f"Shape: {all_trips.shape[0]:,} rows x {all_trips.shape[1]} columns.")

print("\n=== Data Types ===")
print(all_trips.dtypes)

print("\n=== Missing Values ===")
missing = all_trips.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].map(lambda x: f"{x:,}"))
else:
    print("No missing values detected.")

print("\n=== Basic Statistics ===")
print(all_trips.describe(include='all'))

=== Dataset Overview ===
Shape: 3,701,822 rows x 11 columns.

=== Data Types ===
end_station_id                        str
member_casual                         str
start_station_name                    str
start_station_id                      str
started_at            datetime64[us, UTC]
end_station_name                      str
ride_id                               str
ended_at              datetime64[us, UTC]
ride_duration                     float64
started_month                         str
started_day                           str
dtype: object

=== Missing Values ===
No missing values detected.

=== Basic Statistics ===
       end_station_id member_casual                 start_station_name  \
count         3701822       3701822                            3701822   
unique           3244             2                               1855   
top          CHI01747        member  DuSable Lake Shore Dr & Monroe St   
freq            50979       2401191                              3529